In [1]:
from __future__ import print_function
from pyspark import SparkContext
from pyspark.streaming import StreamingContext
from pyspark.sql.session import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.functions import desc
from collections import namedtuple

In [2]:
#sc = SparkContext.getOrCreate()

sc = SparkContext(appName="StreamingTwitterAnalysis")

spark = SparkSession(sc)
ssc = StreamingContext(sc, 10 )
sc.setLogLevel("ERROR")

In [3]:
socket_stream = ssc.socketTextStream("127.0.0.1", 8029)
lines = socket_stream.window( 20 )

print(socket_stream)
print(lines)

In [4]:
hashtags = lines.flatMap( lambda text: text.split( " " ) ).filter( lambda word: word.lower().startswith("#") ).map( lambda 
        word: (word.lower() , 1 ) ).reduceByKey( lambda a, b: a + b )


In [5]:
print(hashtags)

In [6]:
acsds = hashtags.transform(lambda rec: rec.sortBy(lambda x : x[0].lower()).sortBy(lambda x :x[1],ascending=False ))


In [7]:
type(acsds)

pyspark.streaming.dstream.TransformedDStream

pyspark.streaming.dstream.TransformedDStream

# Running GetTweetsStreamData.py

In [8]:
acsds.pprint()

In [9]:
ssc.start()

In [12]:
ssc.awaitTermination(10)

Py4JJavaError: An error occurred while calling o27.awaitTerminationOrTimeout.
: org.apache.spark.SparkException: An exception was raised by Python:
Traceback (most recent call last):
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\util.py", line 68, in call
    r = self.func(t, *rdds)
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\dstream.py", line 668, in <lambda>
    self.func = lambda t, rdd: func(t, prev_func(t, rdd))
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\dstream.py", line 297, in <lambda>
    func = lambda t, rdd: oldfunc(rdd)
  File "C:\Users\DINESH~1\AppData\Local\Temp/ipykernel_7112/598852918.py", line 1, in <lambda>
    acsds = hashtags.transform(lambda rec: rec.sortBy(lambda x : x[0].lower()).sortBy(lambda x :x[1],ascending=False ))
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\rdd.py", line 813, in sortBy
    return self.keyBy(keyfunc).sortByKey(ascending, numPartitions).va

Py4JJavaError: An error occurred while calling o27.awaitTerminationOrTimeout.
: org.apache.spark.SparkException: An exception was raised by Python:
Traceback (most recent call last):
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\util.py", line 68, in call
    r = self.func(t, *rdds)
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\dstream.py", line 668, in <lambda>
    self.func = lambda t, rdd: func(t, prev_func(t, rdd))
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\streaming\dstream.py", line 297, in <lambda>
    func = lambda t, rdd: oldfunc(rdd)
  File "C:\Users\DINESH~1\AppData\Local\Temp/ipykernel_7112/598852918.py", line 1, in <lambda>
    acsds = hashtags.transform(lambda rec: rec.sortBy(lambda x : x[0].lower()).sortBy(lambda x :x[1],ascending=False ))
  File "C:\Users\Dinesh_PC\anaconda3\lib\site-packages\pyspark\rdd.py", line 813, in sortBy
    return self.keyBy(keyfunc).sortByKey(ascending, numPartitions).va